# Lábia — geração do dado de treino (fatia 1 de 2)

O professor (**gpt-oss-20b**, escolhido no piloto) atende os cenários de treino com o **prompt reforçado**, e um cliente simulado da mesma família interpreta o golpista ou o cliente honesto. Cada conversa é julgada pelo verificador e passa pelo **filtro duplo**.

**Amostragem por rejeição:** cada cenário roda uma vez. Os que forem reprovados ganham uma segunda tentativa, com outra semente, enquanto houver tempo no orçamento. Ficamos só com as conversas aprovadas. É como uma prova em que o aluno pode refazer a questão errada, e só as respostas certas entram no gabarito comentado.

Os cenários de treino são divididos em 2 fatias, uma por sessão do Kaggle, para duas GPUs trabalharem ao mesmo tempo. Esta é a fatia 1.

Saídas: todas as conversas (`runs.jsonl`), as aprovadas no formato de SFT (`sft.jsonl`), uma amostra do mesmo tamanho sem filtro, para a ablação (`sft-sem-filtro.jsonl`), e as estatísticas.

In [ ]:
LABIA_COMMIT = "4e001af61eeff09d1946e9f16c64f2c6a5b0404e"
TEACHER = "gpt-oss:20b"
TEACHER_EXTRA = {"reasoning_effort": "low"}
SIM_MAX_TOKENS = 1024
SHARD, NUM_SHARDS = 0, 2
WORKERS = 8
TIME_BUDGET_H = 9.5     # orçamento de horas desta sessão; o Kaggle encerra em 12 h
MAX_ATTEMPTS = 3        # tentativas por cenário (amostragem por rejeição)
MAX_CALLS = 60000       # teto de chamadas da sessão
OUT_DIR = "/kaggle/working/geracao"

## 1. Servidor e modelo

Mesma configuração do piloto, com 8 conversas em paralelo (o piloto mostrou que a memória comporta).

In [ ]:
import os, shutil, subprocess, time, urllib.request, json
SESSION_START = time.perf_counter()
subprocess.run("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-300:])
candidates_dirs = {"/root/.ollama/models": "/", "/tmp/ollama-models": "/tmp"}
models_dir = max(candidates_dirs, key=lambda d: shutil.disk_usage(candidates_dirs[d]).free)
os.makedirs(models_dir, exist_ok=True)
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_CONTEXT_LENGTH="12288", OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="-1", OLLAMA_MODELS=models_dir)
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", TEACHER], env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
r = subprocess.run(["pip", "install", "-q", f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[-300:])

## 2. A fatia de cenários

In [ ]:
from collections import Counter
from labia.scenarios.generator import generate
train = generate("train")
mine = [s for i, s in enumerate(sorted(train, key=lambda s: s.id)) if i % NUM_SHARDS == SHARD]
print(len(mine), "cenários nesta fatia de", len(train))
print("por tática:", dict(Counter(s.tactic.value for s in mine)))

## 3. Rodar com amostragem por rejeição

Rodada 1 para todos. Depois, para os cenários ainda sem conversa aprovada, uma nova tentativa com outra semente, enquanto houver orçamento de tempo. Cada conversa é gravada no arquivo assim que termina: se a sessão cair, nada se perde.

In [ ]:
from pathlib import Path
from labia.agents import LLMAgent, LLMCustomer
from labia.contracts.common import Defense
from labia.dataset import rejection_reasons
from labia.llm import CachedClient, OpenAICompatClient
from labia.runner import Job, load_records, run_batch, run_seed

out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
runs_path = out / "runs.jsonl"
raw = OpenAICompatClient("http://127.0.0.1:11434/v1", "ollama")
agent_client = CachedClient(raw, out / "cache" / "agent", max_calls=MAX_CALLS)
sim_client = CachedClient(raw, out / "cache" / "sim", max_calls=MAX_CALLS)


def approved_ids():
    if not runs_path.exists():
        return set()
    return {r.scenario_id for r in load_records(runs_path) if not rejection_reasons(r)}


for attempt in range(1, MAX_ATTEMPTS + 1):
    elapsed_h = (time.perf_counter() - SESSION_START) / 3600
    if elapsed_h > TIME_BUDGET_H * 0.9:
        print(f"orçamento de tempo quase no fim ({elapsed_h:.1f} h): parando antes da tentativa {attempt}")
        break
    todo = [s for s in mine if s.id not in approved_ids()]
    print(f"tentativa {attempt}: {len(todo)} cenários sem conversa aprovada")
    if not todo:
        break
    run_batch(
        [Job(s, attempt) for s in todo],
        lambda s, r: LLMAgent(agent_client, TEACHER, seed=run_seed(s, r, "agent"), extra=TEACHER_EXTRA),
        lambda s, r: LLMCustomer(sim_client, TEACHER, s, seed=run_seed(s, r, "customer"), extra=TEACHER_EXTRA,
                                 max_tokens=SIM_MAX_TOKENS),
        runs_path, model_id="professor", defense=Defense.REINFORCED, workers=WORKERS, log_every=50,
    )

## 4. Filtro duplo e exportação

- `sft.jsonl`: uma conversa aprovada por cenário (a primeira aprovada), com destilação de contexto: o prompt padrão no lugar do reforçado.
- `sft-sem-filtro.jsonl`: para a **ablação** ("o filtro do verificador importa?"), o mesmo número de conversas, sorteadas entre **todas** as da primeira tentativa, aprovadas ou não. Mesmo tamanho, para a única diferença ser o filtro.

In [ ]:
import random
from labia.dataset import to_sft_example
records = load_records(runs_path)
by_id = {s.id: s for s in mine}
kept, seen = [], set()
for r in sorted(records, key=lambda r: (r.scenario_id, r.run)):
    if r.scenario_id not in seen and not rejection_reasons(r):
        kept.append(r)
        seen.add(r.scenario_id)
first_try = [r for r in records if r.run == 1]
unfiltered = random.Random(SHARD).sample(first_try, min(len(kept), len(first_try)))


def dump(rows, name):
    with open(out / name, "wb") as fh:
        for r in rows:
            fh.write((json.dumps(to_sft_example(r, by_id[r.scenario_id]), ensure_ascii=False) + "\n").encode())


dump(kept, "sft.jsonl")
dump(unfiltered, "sft-sem-filtro.jsonl")
reasons = Counter(reason for r in first_try for reason in rejection_reasons(r))
hours = (time.perf_counter() - SESSION_START) / 3600
stats = {
    "fatia": SHARD, "professor": TEACHER, "cenarios": len(mine), "conversas": len(records),
    "aprovadas": len(kept), "cobertura": round(len(kept) / len(mine), 3),
    "aprovacao_primeira_tentativa": round(sum(not rejection_reasons(r) for r in first_try) / max(1, len(first_try)), 3),
    "motivos_reprovacao_primeira_tentativa": dict(reasons.most_common()),
    "vereditos": dict(Counter(r.verdict.value for r in records)),
    "chamadas_validas": round(sum(r.n_valid_calls for r in records) / max(1, sum(r.n_calls for r in records)), 4),
    "horas_de_sessao": round(hours, 2), "chamadas_ao_modelo": agent_client.calls_made + sim_client.calls_made,
    "aprovadas_por_tatica": dict(Counter(r.tactic.value for r in kept)),
    "aprovadas_por_lado": dict(Counter("legitimo" if r.legit else "golpe" for r in kept)),
}
(out / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.rmtree(out / "cache", ignore_errors=True)  # o cache é grande e fica só na sessão
print(json.dumps(stats, ensure_ascii=False, indent=2))